> <p><small>This notebook is made available subject to the licence and terms set out in <a href="https://creativecommons.org/licenses/by/4.0">https://creativecommons.org/licenses/by/4.0</a>.</small></p>

<img src="https://pub-bba109a9a6ac49e3b428cdca19c34363.r2.dev/LT%20-%20Session%203.jpg">

# 3.7 AI Long Activity 2 - Lab: Neural Network Training (Teacher)

Train a small neural network on a simple dataset and observe how overfitting occurs.

60 minutes

## Overview

This notebook is designed to make overfitting visible.

Students train and compare:
- A high-capacity neural network.
- A smaller neural network.
- A regularized high-capacity neural network.

The notebook combines:
- Rraining and test accuracy.
- Decision boundary visualisations.
- Loss curves.
- A simple gradient descent example.

> ℹ️ **Info:**
>
> The dataset and regularization setting have been chosen so that the regularized model has a visible effect relative to the unregularized large model. Exact accuracies may vary slightly by environment.

## Your task

Use this notebook to guide students through the lab.

1. Demonstrate how the dataset is generated.
2. Explain why the train/test split matters.
3. Show how decision boundary plots make overfitting visible.
4. Compare the high-capacity, smaller, and regularized models.
5. Use the experiment block to vary one setting at a time.
6. Connect the training process to gradient descent.

## Step 1 — Imports and setup

This cell imports the libraries needed for the lab.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import make_moons
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

## Step 2 — Generate a noisy dataset

The `make_moons` dataset is used because it is visually simple but non-linear.

> 📝 **Teacher notes:**
> - The dataset is small and noisy enough for overfitting to appear.
> - The two-dimensional input makes the decision boundary easy to plot.

In [ ]:
X, y = make_moons(
    n_samples=150,
    noise=0.3,
    random_state=42,
)

plt.figure(figsize=(6, 4))
plt.scatter(
    X[:, 0],
    X[:, 1],
    c=y,
    cmap="coolwarm",
    edgecolor="k",
)

plt.title("Noisy moon dataset")
plt.xlabel("x1")
plt.ylabel("x2")
plt.show()

## Step 3 — Train/test split

This step separates data used for learning from data used for evaluation.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
)

print("Training size:", len(X_train))
print("Test size:", len(X_test))

## Step 4 — Helper function to visualize decision boundaries

Accuracy numbers are useful, but they do not always show why a model generalizes badly. This function displays the classification regions learned by the model.

In [ ]:
def plot_decision_boundary(model, X, y, title):
    """Plot a model's predicted decision regions together with data points.

    Args:
        model: A fitted scikit-learn classifier with a predict method.
        X: A two-column array containing input features.
        y: A one-dimensional array containing class labels.
        title: Title to display above the plot.
    """
    x_min = X[:, 0].min() - 0.5
    x_max = X[:, 0].max() + 0.5
    y_min = X[:, 1].min() - 0.5
    y_max = X[:, 1].max() + 0.5

    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 300),
        np.linspace(y_min, y_max, 300),
    )

    grid = np.c_[xx.ravel(), yy.ravel()]
    predictions = model.predict(grid)
    predictions = predictions.reshape(xx.shape)

    plt.figure(figsize=(6, 4))
    plt.contourf(
        xx,
        yy,
        predictions,
        alpha=0.25,
        cmap="coolwarm",
    )
    plt.scatter(
        X[:, 0],
        X[:, 1],
        c=y,
        cmap="coolwarm",
        edgecolor="k",
    )

    plt.title(title)
    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.show()

## Step 5 — Model 1: high-capacity network

This model has enough flexibility to fit the training data very closely. With a small noisy dataset, it may also fit noise.

In [ ]:
model_big = MLPClassifier(
    hidden_layer_sizes=(100, 100),
    max_iter=2000,
    random_state=42,
)

model_big.fit(X_train, y_train)

train_acc_big = accuracy_score(
    y_train,
    model_big.predict(X_train),
)

test_acc_big = accuracy_score(
    y_test,
    model_big.predict(X_test),
)

print("High-capacity model")
print("Training accuracy:", train_acc_big)
print("Test accuracy:", test_acc_big)

plot_decision_boundary(
    model_big,
    X_train,
    y_train,
    "High-capacity model: training data",
)

plot_decision_boundary(
    model_big,
    X_test,
    y_test,
    "High-capacity model: test data",
)

## Step 6 — Observe training loss

The attribute `loss_curve_` shows how the training loss changes during optimization.

In [ ]:
print("Loss curve (first 10 values):")
print(model_big.loss_curve_[:10], "...")

plt.figure(figsize=(6, 4))
plt.plot(model_big.loss_curve_)

plt.title("Training loss over iterations")
plt.xlabel("Iteration")
plt.ylabel("Loss")
plt.show()

## Step 7 — Model 2: smaller network

A smaller model cannot fit arbitrary detail as easily. This can reduce overfitting, although it may also underfit if the model is too simple.

In [ ]:
model_small = MLPClassifier(
    hidden_layer_sizes=(20,),
    max_iter=2000,
    random_state=42,
)

model_small.fit(X_train, y_train)

train_acc_small = accuracy_score(
    y_train,
    model_small.predict(X_train),
)

test_acc_small = accuracy_score(
    y_test,
    model_small.predict(X_test),
)

print("Smaller model")
print("Training accuracy:", train_acc_small)
print("Test accuracy:", test_acc_small)

plot_decision_boundary(
    model_small,
    X_test,
    y_test,
    "Smaller model: test data",
)

## Step 8 — Model 3: regularized network

This model uses the same large architecture as Model 1, but adds stronger L2 regularization through `alpha=0.1`.

> 📝 **Teacher notes:**
> - The previous value `alpha=0.01` did not show a clear enough difference.
> - The current setting is intended to make the regularization effect visible.

In [ ]:
model_reg = MLPClassifier(
    hidden_layer_sizes=(100, 100),
    alpha=0.1,
    max_iter=2000,
    random_state=42,
)

model_reg.fit(X_train, y_train)

train_acc_reg = accuracy_score(
    y_train,
    model_reg.predict(X_train),
)

test_acc_reg = accuracy_score(
    y_test,
    model_reg.predict(X_test),
)

print("Regularised large model")
print("Training accuracy:", train_acc_reg)
print("Test accuracy:", test_acc_reg)

plot_decision_boundary(
    model_reg,
    X_test,
    y_test,
    "Regularised large model: test data",
)

## Step 9 — Scaffolded experiment block

Students can change one setting at a time and observe the result.

In [ ]:
hidden = (50, 50)
alpha = 0.01
max_iter = 2000

model_exp = MLPClassifier(
    hidden_layer_sizes=hidden,
    alpha=alpha,
    max_iter=max_iter,
    random_state=42,
)

model_exp.fit(X_train, y_train)

train_acc_exp = accuracy_score(
    y_train,
    model_exp.predict(X_train),
)

test_acc_exp = accuracy_score(
    y_test,
    model_exp.predict(X_test),
)

print("Experiment model")
print("hidden_layer_sizes =", hidden)
print("alpha =", alpha)
print("max_iter =", max_iter)
print("Training accuracy:", train_acc_exp)
print("Test accuracy:", test_acc_exp)

plot_decision_boundary(
    model_exp,
    X_test,
    y_test,
    "Experiment model: test data",
)

## Step 10 — Summary comparison

This table helps compare models directly.

In [ ]:
print("Summary")
print("-------------------------------")

print(
    f"High-capacity model   | "
    f"train = {train_acc_big:.3f} | "
    f"test = {test_acc_big:.3f}"
)

print(
    f"Smaller model         | "
    f"train = {train_acc_small:.3f} | "
    f"test = {test_acc_small:.3f}"
)

print(
    f"Regularised model     | "
    f"train = {train_acc_reg:.3f} | "
    f"test = {test_acc_reg:.3f}"
)

print(
    f"Experiment model      | "
    f"train = {train_acc_exp:.3f} | "
    f"test = {test_acc_exp:.3f}"
)

## Step 11 — What does training actually do?

The simple gradient descent example below illustrates the same principle used in neural network training.

In [ ]:
def loss(w):
    """Return the value of L(w) = (w - 3)^2."""
    return (w - 3) ** 2


def grad(w):
    """Return the gradient of L(w) = (w - 3)^2."""
    return 2 * (w - 3)


w = 0.0
eta = 0.1
trajectory = [w]

for _ in range(10):
    w = w - eta * grad(w)
    trajectory.append(w)

print("Parameter updates:", trajectory)

plt.figure(figsize=(6, 4))
plt.plot(
    trajectory,
    marker="o",
)
plt.title("Gradient descent towards minimum")
plt.xlabel("Step")
plt.ylabel("w")
plt.show()

> 💭 **Reflection:**
> Students should compare:
> 1. Which model had the highest training accuracy?
> 2. Which model had the best test accuracy?
> 3. Which decision boundary looked most likely to overfit?
> 4. Why can a regularized model outperform a larger unregularized model on new data?
>
> **Key takeaway**
>
> A good model is not the one that fits the training data best. A good model is the one that generalizes best.